## Structured Output

Models can be instructed to return responses matching a specific schema instead of free-form text. This is useful when you need reliable, machine-readable outputs for downstream code or APIs.

LangChain supports two main ways to enforce structured output:
1. **`model.with_structured_output(Schema)`**: Binds a schema directly to a chat model.
2. **`create_agent(..., response_format=Schema)`**: Enforces structured output on an agent, storing the parsed result in `result["structured_response"]`.

Supported schema types include **Pydantic `BaseModel`**, **`TypedDict`**, and Python **`@dataclass`**.

### Pydantic (`BaseModel`)

Pydantic models provide the richest feature set, including runtime type validation, field `description`s (which guide the LLM), default values, and nested structures.

In [1]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

# Load environment variables and initialize the base chat model
load_dotenv()
model = init_chat_model("groq:qwen/qwen3.8-27b")

In [2]:
from pydantic import BaseModel, Field

# Define the output schema; Field(description=...) tells the LLM what each field represents
class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    year: int = Field(description="This is the year the movie was released")
    director: str = Field(description="The director of the movie")
    rating: float = Field(description="The rating of the movie out of 10")

In [4]:
# Wrap the model so invoke() returns a validated Movie Pydantic instance
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.8.4', 'langchain': '1.5.2', 'langchain-groq': '1.2.1'}}, profile={'name': 'Qwen3.8 27B', 'status': 'beta', 'release_date': '2026-08-14', 'last_updated': '2026-08-14', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x112a44440>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x112a44ec0>, model_name='qwen/qwen3.8-27b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title'

In [5]:
# Invoke the structured model; the response is parsed directly into a Movie object
response = model_with_structure.invoke("PRovide details about the movie Inception by Chris Nolan")
response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

### Raw Message Output Alongside Parsed Structure (`include_raw=True`)

Passing `include_raw=True` to `with_structured_output()` returns a dictionary with:
- `"raw"`: The original `AIMessage` (including token usage and metadata)
- `"parsed"`: The validated schema instance
- `"parsing_error"`: Any error encountered during parsing (or `None`)

In [7]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="This is the year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The rating of the movie out of 10")

# include_raw=True returns {"raw": AIMessage, "parsed": Movie, "parsing_error": None}
model_with_structure = model.with_structured_output(Movie, include_raw=True)

response = model_with_structure.invoke("Provide details about the movie Shrek")
response

{'raw': AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '185tqvxx3', 'function': {'arguments': '{"director":"Andrew Adamson, Vicky Jenson","rating":8.3,"title":"Shrek","year":2001}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 74, 'prompt_tokens': 353, 'total_tokens': 427, 'completion_time': 0.19822726, 'completion_tokens_details': None, 'prompt_time': 0.025429368, 'prompt_tokens_details': None, 'queue_time': 0.074769119, 'total_time': 0.223656628}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_21e59ac2de', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a20c15-2bee-702c-91d3-646401b27278-0', tool_calls=[{'name': 'Movie', 'args': {'director': 'Andrew Adamson, Vicky Jenson', 'rating': 8.3, 'title': 'Shrek', 'year': 2001}, 'id': '185tqvxx3', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 353, 'output_tokens

### Nested Structures

Pydantic models can reference other Pydantic models (e.g., `list[Actor]`) and optional fields (`float | None`) to extract complex hierarchical data.

In [9]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]  # Nested list of Actor objects
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")  # Optional field defaulting to None

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie The Wolf Of Wall Street")
response

MovieDetails(title='The Wolf of Wall Street', year=2013, cast=[Actor(name='Leonardo DiCaprio', role='Jordan Belfort'), Actor(name='Jonah Hill', role='Donnie Azoff'), Actor(name='Margot Robbie', role='Naomi Lapaglia'), Actor(name='Matthew McConaughey', role='Mark Hunter'), Actor(name='Kyle Chandler', role='Dickie J. Bonner'), Actor(name='Rob Reiner', role='C. Michael (Broker)')], genres=['Biography', 'Comedy', 'Crime', 'Drama'], budget=100.0)

### `TypedDict`

`TypedDict` provides a lightweight alternative using Python's built-in typing (returning a plain `dict` instead of a Pydantic object). Use `Annotated[type, ..., "description"]` to attach field descriptions when you don't need Pydantic runtime validation.

In [11]:
from typing_extensions import TypedDict, Annotated

# TypedDict with Annotated metadata for field descriptions
class movieDict(TypedDict):
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year of the movie release"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The rating of the movie out of 10"]

# Returns a standard Python dictionary matching movieDict
model_with_typeddict = model.with_structured_output(movieDict)
response = model_with_typeddict.invoke("Provide details of the movie avengers age of ultron")
response

{'director': 'Joss Whedon',
 'rating': 7.3,
 'title': 'Avengers: Age of Ultron',
 'year': 2015}

In [14]:
# Nested TypedDict example
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie The Wolf Of Wall Street")
response

{'budget': 100000000,
 'cast': [{'name': 'Leonardo DiCaprio', 'role': 'Jordan Belfort'},
  {'name': 'Margot Robbie', 'role': 'Naomi Lapaglia'},
  {'name': 'Jonah Hill', 'role': 'Donnie Azoff'},
  {'name': 'Matthew McConaughey', 'role': 'Mark Hanna'},
  {'name': 'Kyle Chandler', 'role': 'FBI Agent Robert Levinson'},
  {'name': 'Rob Reiner', 'role': 'Jean-Baptiste'},
  {'name': 'Jon Favreau', 'role': 'Sam Rabkin'},
  {'name': 'Jean Dujardin', 'role': 'Bradka'},
  {'name': 'Jack Huston', 'role': 'Danny Porush'},
  {'name': "Denis O'Hare", 'role': 'Curtis Kull'}],
 'genres': ['Biography', 'Crime', 'Drama', 'Comedy'],
 'title': 'The Wolf of Wall Street',
 'year': 2013}

In [15]:
# Inspect the model's capability profile (e.g., structured output, tool calling, context limits)
model.profile

{'name': 'Qwen3.8 27B',
 'status': 'beta',
 'release_date': '2026-08-14',
 'last_updated': '2026-08-14',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 16384,
 'text_inputs': True,
 'image_inputs': True,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': True,
 'temperature': True}

### Structured Output in Agents (`response_format`) & `@dataclass`

When building an agent with `create_agent()`, pass `response_format=Schema` to enforce structured output at the end of the agent's run. LangChain auto-selects the best strategy (e.g., `ProviderStrategy`) and stores the parsed object under `result["structured_response"]`.

This works with **Pydantic `BaseModel`**, **`TypedDict`**, and Python **`@dataclass`**.

In [ ]:
# Using Pydantic BaseModel with create_agent(..., response_format=...)
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model="groq:qwen/qwen3.8-27b",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, Male , john@example.com, (555) 123-4567"}]
})

# Access the validated ContactInfo instance from result["structured_response"]
result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [19]:
# Using TypedDict with create_agent(..., response_format=...)
from typing_extensions import TypedDict
from langchain.agents import create_agent

class ContactInfo(TypedDict):
    """Contact information for a person."""
    name: str   # The name of the person
    email: str  # The email address of the person
    phone: str  # The phone number of the person

agent = create_agent(
    model="groq:qwen/qwen3.8-27b",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

# Returns a dictionary matching ContactInfo
result["structured_response"]

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

In [20]:
# Using Python @dataclass with create_agent(..., response_format=...)
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str   # The name of the person
    email: str  # The email address of the person
    phone: str  # The phone number of the person

agent = create_agent(
    model="groq:qwen/qwen3.8-27b",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

# Returns a ContactInfo dataclass instance
result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')